# Sesión 7 · Actividad 6
# Negative Prompt: indicar a la IA qué debe evitar

**Curso:** Introducción a la Inteligencia Artificial Generativa  
**Modalidad:** laboratorio guiado en Google Colab

---

## Propósito

En esta actividad experimentarás con el **negative prompt** para descubrir:

- cómo orientar al modelo para evitar ciertos rasgos;
- qué ocurre cuando se agregan restricciones gradualmente;
- si un negative prompt siempre mejora una imagen;
- qué riesgos aparecen cuando se incluyen demasiadas instrucciones negativas.

> **Regla experimental:** mantendremos fija la semilla y los parámetros principales.  
> Solo cambiaremos el negative prompt.


## La idea central es:

> **El prompt positivo indica qué favorecer; el negative prompt indica qué reducir o evitar.**

### Preguntas iniciales

- ¿La IA puede comprender una instrucción como “no dibujes manos deformes”?
- ¿Eliminar defectos es tan sencillo como enumerarlos?
- ¿Un negative prompt muy largo podría perjudicar el resultado?



# 0. Preparación del entorno

En Google Colab selecciona:

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**

Después ejecuta las celdas en orden.

Este cuaderno es completamente independiente y no requiere ninguna actividad anterior.


In [ ]:
import torch

print("Versión de PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No se detectó una GPU. Selecciona: "
        "Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU."
    )


# 1. Configuración del guardado

Puedes guardar los resultados:

- temporalmente en Colab;
- permanentemente en Google Drive.

Para guardar en Drive cambia `USAR_DRIVE` a `True`.


In [ ]:
from pathlib import Path

USAR_DRIVE = False

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_RESULTADOS = Path(
        "/content/drive/MyDrive/IA_Generativa/Sesion7/Actividad6_NegativePrompt"
    )
else:
    CARPETA_RESULTADOS = Path("/content/Actividad6_NegativePrompt")

CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)
print("Los resultados se guardarán en:", CARPETA_RESULTADOS)


# 2. Instalación de bibliotecas

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors


# 3. Carga del modelo

Usaremos Stable Diffusion 1.5 con DPM-Solver++.

> **Nota del profesor:** la misma semilla se conservará en todas las generaciones para facilitar una comparación visual justa.


In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt

from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
    safety_checker=None
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="dpmsolver++"
)

pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

print("Modelo cargado correctamente.")


# 4. Diseño del experimento

Mantendremos constantes:

- **Prompt positivo:** retrato cinematográfico de una astronauta;
- **Semilla:** 42;
- **Pasos:** 25;
- **Guidance Scale:** 7.5;
- **Tamaño:** 512 × 512 píxeles;
- **Scheduler:** DPM-Solver++.

Probaremos cinco niveles:

1. sin negative prompt;
2. defectos visuales generales;
3. defectos anatómicos;
4. defectos técnicos y de composición;
5. lista excesivamente larga.


In [ ]:
PROMPT = (
    "A cinematic portrait of a female astronaut inside a spacecraft, "
    "realistic photography, dramatic lighting, highly detailed"
)

SEED = 42
STEPS = 25
GUIDANCE = 7.5
WIDTH = 512
HEIGHT = 512

NEGATIVE_PROMPTS = {
    "Sin negativo": "",
    "Calidad básica": (
        "blurry, low quality, low resolution, noisy, out of focus"
    ),
    "Anatomía": (
        "blurry, low quality, deformed face, malformed eyes, "
        "bad anatomy, extra fingers, missing fingers"
    ),
    "Técnico": (
        "blurry, low quality, deformed face, bad anatomy, "
        "cropped, duplicate, text, watermark, oversaturated"
    ),
    "Excesivo": (
        "blurry, low quality, low resolution, noisy, out of focus, "
        "deformed face, asymmetrical eyes, malformed eyes, bad anatomy, "
        "extra fingers, missing fingers, extra limbs, duplicate, cropped, "
        "text, watermark, logo, oversaturated, undersaturated, dark, bright, "
        "simple background, complex background, dramatic, boring, artificial, "
        "unrealistic, painting, illustration, cartoon, 3d render"
    )
}

def generar_imagen(nombre, negative_prompt):
    generator = torch.Generator(device="cuda").manual_seed(SEED)
    inicio = time.time()

    imagen = pipe(
        prompt=PROMPT,
        negative_prompt=negative_prompt,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=WIDTH,
        height=HEIGHT,
        generator=generator
    ).images[0]

    duracion = time.time() - inicio

    nombre_archivo = (
        nombre.lower()
        .replace(" ", "_")
        .replace("í", "i")
        .replace("á", "a")
        .replace("é", "e")
    )
    imagen.save(CARPETA_RESULTADOS / f"{nombre_archivo}.png")

    return imagen, duracion


# 5. Predicción

Antes de ejecutar responde:

1. ¿Qué imagen crees que tendrá mejor calidad?
2. ¿Cuál negative prompt podría producir el mayor cambio?
3. ¿El negative prompt excesivo mejorará o perjudicará la imagen?
4. ¿Qué elementos podrían desaparecer sin que fuera nuestra intención?


# 6. Generación progresiva

> **Observar no solo si desaparecen defectos, sino también si cambian:**
>
> - rostro;
> - encuadre;
> - iluminación;
> - fondo;
> - estilo;
> - identidad visual del personaje.


In [ ]:
imagenes = []
tiempos = []
nombres = []

for nombre, negativo in NEGATIVE_PROMPTS.items():
    print(f"Generando: {nombre}...")
    imagen, duracion = generar_imagen(nombre, negativo)
    nombres.append(nombre)
    imagenes.append(imagen)
    tiempos.append(duracion)

print("Generación terminada.")


# 7. Comparación visual

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for ax, imagen, nombre in zip(axes, imagenes, nombres):
    ax.imshow(imagen)
    ax.set_title(nombre)
    ax.axis("off")

plt.tight_layout()

ruta = CARPETA_RESULTADOS / "comparacion_negative_prompts.png"
plt.savefig(ruta, dpi=150, bbox_inches="tight")
plt.show()

print("Comparación guardada en:", ruta)


# 8. Registro de observaciones

Evalúa cada imagen en una escala de 1 a 5.

| Configuración | Calidad | Anatomía | Naturalidad | Fidelidad al prompt | Cambios no deseados | Observaciones |
|---|---:|---:|---:|---:|---:|---|
| Sin negativo | | | | | | |
| Calidad básica | | | | | | |
| Anatomía | | | | | | |
| Técnico | | | | | | |
| Excesivo | | | | | | |


# 9. Comparación de tiempos

In [ ]:
tabla_tiempos = pd.DataFrame({
    "Configuración": nombres,
    "Tiempo de generación (s)": [round(t, 2) for t in tiempos]
})

tabla_tiempos


# 10. Preguntas de análisis

### 1.
¿Qué cambió al incorporar el primer negative prompt?

### 2.
¿La lista de defectos anatómicos resolvió todos los problemas?

### 3.
¿Qué configuración produjo la imagen más natural?

### 4.
¿El negative prompt excesivo mejoró la calidad?

### 5.
¿Qué términos negativos parecieron influir también en la composición o el estilo?

### 6.
¿El negative prompt funcionó como una prohibición absoluta?

### 7.
¿Cambió significativamente el tiempo de generación?

### 8.
Formula una conclusión:

> Un negative prompt permite...


## Reflexión:
> “El negative prompt no borra objetos como una herramienta de edición. Reorienta el proceso de generación para alejarlo de ciertos conceptos.”


# 11. Experimento focalizado: evitar texto

Ahora usaremos una escena donde el modelo podría intentar crear letras o anuncios.

Compararemos:

- sin negative prompt;
- con `text, letters, words, watermark, logo`.


In [ ]:
PROMPT_TEXTO = (
    "A futuristic street market at night, neon signs, cinematic photography, "
    "highly detailed"
)

def generar_escena_texto(negative_prompt, nombre):
    generator = torch.Generator(device="cuda").manual_seed(700)

    imagen = pipe(
        prompt=PROMPT_TEXTO,
        negative_prompt=negative_prompt,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=WIDTH,
        height=HEIGHT,
        generator=generator
    ).images[0]

    imagen.save(CARPETA_RESULTADOS / f"{nombre}.png")
    return imagen

imagen_sin_restriccion = generar_escena_texto("", "mercado_sin_negativo")
imagen_sin_texto = generar_escena_texto(
    "text, letters, words, watermark, logo, signature",
    "mercado_sin_texto"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(imagen_sin_restriccion)
axes[0].set_title("Sin negative prompt")
axes[0].axis("off")

axes[1].imshow(imagen_sin_texto)
axes[1].set_title("Evitando texto")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## Reflexión del experimento focalizado

- ¿Desaparecieron todas las formas parecidas a letras?
- ¿Se modificaron los letreros o la arquitectura?
- ¿El modelo sustituyó el texto por otros elementos?
- ¿La imagen mejoró para el propósito planteado?


# 12. Mini reto: diseña un negative prompt útil

Usa este prompt positivo:

> `A professional product photograph of a luxury wristwatch on a reflective surface`

Diseña un negative prompt de **máximo ocho conceptos**.

Tu objetivo es:

- conservar el producto;
- evitar defectos;
- mantener un aspecto profesional;
- no destruir la composición.


In [ ]:
PROMPT_RETO = (
    "A professional product photograph of a luxury wristwatch "
    "on a reflective surface, studio lighting, highly detailed"
)

# Máximo ocho conceptos separados por comas
NEGATIVE_RETO = (
    "blurry, low quality, deformed, duplicate, text, watermark, cropped, noisy"
)

generator = torch.Generator(device="cuda").manual_seed(2026)

imagen_reto = pipe(
    prompt=PROMPT_RETO,
    negative_prompt=NEGATIVE_RETO,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE,
    width=WIDTH,
    height=HEIGHT,
    generator=generator
).images[0]

ruta_reto = CARPETA_RESULTADOS / "mini_reto_negative_prompt.png"
imagen_reto.save(ruta_reto)

display(imagen_reto)
print("Negative prompt utilizado:", NEGATIVE_RETO)


## Reflexión del mini reto

- Conceptos negativos utilizados:
- Razón para incluirlos:
- Problemas que intentabas evitar:
- Resultado obtenido:
- Cambios no deseados:
- ¿Qué término eliminarías o agregarías?


# 13. Evidencia de aprendizaje

Entrega:

1. comparación de los cinco niveles;
2. tabla de observaciones;
3. respuestas de análisis;
4. comparación del experimento para evitar texto;
5. imagen y negative prompt del mini reto;
6. conclusión personal de máximo 100 palabras.


# 14. Cierre conceptual

El negative prompt:

- no edita directamente una imagen;
- reduce la probabilidad de ciertos conceptos;
- no funciona como prohibición absoluta;
- puede mejorar algunos resultados;
- puede perjudicar otros si contiene contradicciones;
- debe ser específico y moderado.

> **Un buen negative prompt no es el más largo: es el más pertinente.**
